# Face recognition and authentication — complete workflow
**Owner: Maurice · Branch: `feature/face`**

This notebook builds a face-only authentication model for the multimodal assignment.
It performs data auditing, image visualization, filters, source-level splitting,
training-only augmentation, pretrained embedding extraction, baseline comparison,
validation, open-set testing, model export and reload verification.

Run cells from top to bottom. CPU is sufficient. The accompanying `face_auth/`
package is required and keeps inference identical to training. Raw data are never modified.

## 1. Environments and files
- **Local / Anaconda:** `conda env create -f environment-face.yml`, then
  `conda activate multimodal-face` and `python -m ipykernel install --user --name multimodal-face`.
  Select that kernel and open this notebook from Jupyter or VS Code.
- **Colab:** upload/clone the **whole repository** under `/content`, including
  `face_auth`, or mount Drive and set `PROJECT_ROOT_OVERRIDE` below.
- **Kaggle:** attach the repository and images as private datasets; set paths below
  if automatic discovery is ambiguous. Outputs go to `/kaggle/working/multimodal-face`,
  not the read-only input dataset. Enable Internet for first-time model downloads,
  or attach both ONNX files as an input dataset and set `WEIGHTS_DIR_OVERRIDE`.

Use consenting participants' data. Images must be `images/<user_id>/<file>`. Put
separate unknown people in `images/unauthorized/` or `images/unknown/`.


### 1.1 Configure this run

Choose project, image, output and weight directories. Leave overrides as `None` for automatic local discovery. The seed fixes split reproducibility; installation and model downloads remain explicit.

In [ ]:
# Configuration: edit only this cell for your machine / cloud dataset.
from pathlib import Path
import os

PROJECT_ROOT_OVERRIDE = None  # e.g. "/content/multimodal_authentication_preprocessing"
IMAGE_DIR_OVERRIDE = None     # e.g. "/kaggle/input/private-faces/images"
OUTPUT_ROOT_OVERRIDE = None   # writable destination; default is project root locally
WEIGHTS_DIR_OVERRIDE = None   # offline folder containing both ONNX model files
DOWNLOAD_WEIGHTS = True       # set False when running offline
SEED = 42
CONFIDENCE_THRESHOLD = 0.5     # fixed before validation/test, not tuned on test
INSTALL_MISSING = False       # True only if installing into the CURRENT kernel is desired

### 1.2 Check the active kernel and dependencies

Check packages in the kernel executing this notebook. Enable `INSTALL_MISSING` only when you want that kernel to install missing packages. In Anaconda, selecting the correct kernel comes before running this cell.

In [ ]:
# Never activate conda inside a notebook; install into the selected kernel.
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.2,<3",
    "sklearn": "scikit-learn>=1.4,<2", "joblib": "joblib>=1.3,<2",
    "PIL": "pillow>=10,<13", "matplotlib": "matplotlib>=3.8,<4",
    "cv2": "opencv-python-headless>=4.8,<5",
}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError(f"Missing {missing}. Install requirements/face.txt into this kernel, "
                           "or set INSTALL_MISSING=True and rerun. Restart after installing.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Kernel:", sys.executable)

### 1.3 Resolve portable input and output directories

Locate the full repository and create writable output folders. Kaggle outputs go to its working directory. Cloud datasets can stay read-only while processed images, reports and models are saved elsewhere.

In [ ]:
# Portable project discovery; the repository package must accompany the notebook.
candidates = [Path.cwd(), *Path.cwd().parents]
for base in (Path("/content"), Path("/kaggle/input")):
    if base.exists():
        candidates += list(base.glob("*")) + list(base.glob("*/*"))
if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    matches = [p for p in candidates if (p / "face_auth/embeddings.py").is_file()]
    if not matches:
        raise FileNotFoundError("Upload/clone the full repository, then set PROJECT_ROOT_OVERRIDE")
    PROJECT_ROOT = matches[0].resolve()
if not (PROJECT_ROOT / "face_auth/embeddings.py").is_file():
    raise FileNotFoundError("PROJECT_ROOT must contain the face_auth package")
sys.path.insert(0, str(PROJECT_ROOT))

IMAGE_DIR = Path(IMAGE_DIR_OVERRIDE).expanduser() if IMAGE_DIR_OVERRIDE else PROJECT_ROOT / "data/raw/images"
OUTPUT_ROOT = (Path(OUTPUT_ROOT_OVERRIDE).expanduser() if OUTPUT_ROOT_OVERRIDE else
               Path("/kaggle/working/multimodal-face") if Path("/kaggle/working").exists() else PROJECT_ROOT)
PROCESSED = OUTPUT_ROOT / "data/processed"
MODELS = OUTPUT_ROOT / "models"
REPORTS = OUTPUT_ROOT / "reports/face"
WEIGHTS_DIR = Path(WEIGHTS_DIR_OVERRIDE).expanduser() if WEIGHTS_DIR_OVERRIDE else MODELS / "pretrained"
for directory in (PROCESSED, MODELS, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Set IMAGE_DIR_OVERRIDE to your image directory: {IMAGE_DIR}")
print("Project:", PROJECT_ROOT, "\nImages:", IMAGE_DIR, "\nOutputs:", OUTPUT_ROOT)

### 1.4 Import the analysis tools and record versions

Load plotting, data analysis, image encoding and training utilities. Record Python/library versions and initialize reproducibility settings before inspecting or fitting any data.

In [ ]:
import json
import platform
from datetime import datetime, timezone
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import cv2
from PIL import Image
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, log_loss, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

from face_auth.embeddings import FaceEncoder, FaceInputError, ensure_weights, read_rgb, sha256
from face_auth.pipeline import (inventory, split_sources, detect_inventory, build_features,
                                image_variants, FaceAuthenticator, closed_metrics,
                                calibrate_similarity)
np.random.seed(SEED)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": False})
versions = {p: importlib.metadata.version(p) for p in
            ("numpy", "pandas", "scikit-learn", "joblib", "pillow", "matplotlib")}
versions["opencv"] = cv2.__version__
print("Python", platform.python_version(), "|", versions)

## 2. Inventory, quality and duplicates
Audit dimensions, brightness, contrast, decoding failures and exact decoded-pixel
duplicates. Identical images cannot appear in multiple splits. Review visually
for near-duplicates and same-session captures; a hash cannot detect those reliably.
For a larger dataset, add session IDs and split by acquisition session.

A class needs **at least three independent valid originals** to be enrolled here:
one or more training captures, one validation capture and one test capture.
Underrepresented identities are held out entirely as unknown test users. This
policy makes the current small dataset executable without reusing photographs
across partitions; it does not establish robust real-world performance.


In [ ]:
audit = inventory(IMAGE_DIR)
display(audit[["user_id", "width", "height", "brightness", "contrast", "status"]])
display(pd.crosstab(audit.user_id, audit.status))
valid = audit[audit.status == "ok"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
valid.groupby("user_id").size().plot.bar(ax=axes[0], title="Independent images per identity")
axes[1].scatter(valid.width, valid.height)
axes[1].set(xlabel="Width (pixels)", ylabel="Height (pixels)", title="Image dimensions")
axes[2].scatter(valid.brightness, valid.contrast)
axes[2].set(xlabel="Mean grayscale brightness", ylabel="Grayscale standard deviation", title="Exposure / contrast")
plt.tight_layout()
plt.savefig(REPORTS / "data_quality.png", dpi=140, bbox_inches="tight")
plt.show()

# Show every member's available expressions, not just the first image.
fig, axes = plt.subplots(len(valid), 1, figsize=(6, 3 * len(valid)), squeeze=False)
for axis, row in zip(axes.flat, valid.itertuples()):
    axis.imshow(read_rgb(row.path))
    axis.set_title(f"{row.user_id}: {Path(row.path).name}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Image filters and augmentations
Display rotation, horizontal reflection, grayscale conversion, brightness,
contrast, Gaussian smoothing and sharpening. Filters reveal sensitivity to
lighting and texture; not every filter necessarily improves recognition.
Only training crops will receive these transforms. Validation/test keep one
unaltered original each. Avoid vertical flips and extreme transformations.


In [ ]:
sample = read_rgb(valid.iloc[0].path)
variants = image_variants(sample)
fig, axes = plt.subplots(3, 3, figsize=(11, 10))
for axis, (name, image) in zip(axes.flat, variants.items()):
    axis.imshow(image)
    axis.set_title(name.replace("_", " "))
    axis.axis("off")
plt.tight_layout()
plt.savefig(REPORTS / "image_filters.png", dpi=140, bbox_inches="tight")
plt.show()

## 4. Pretrained detection, alignment and embeddings
Use OpenCV Zoo **YuNet (2023mar)** for face detection/landmarks and
**SFace (2021dec)** for a 128-dimensional pretrained identity embedding.
The original image is EXIF-corrected, converted to RGB, resized only if its
longest side exceeds 1200 pixels, then converted to OpenCV BGR. Exactly one
face must pass detection confidence 0.8 with a minimum 30-pixel box side.
SFace aligns facial landmarks to a 112×112 crop. Each embedding is L2-normalized.

Downloads occur only if `DOWNLOAD_WEIGHTS=True` and files are missing. In offline
Kaggle/Colab, supply both ONNX files. Record SHA-256 values for reproducibility;
upstream download URLs are not immutable, so archive the exact weights used.
Review the upstream model licenses before redistributing them.


In [ ]:
weight_paths = ensure_weights(WEIGHTS_DIR, download=DOWNLOAD_WEIGHTS)
weight_hashes = {name: sha256(path) for name, path in weight_paths.items()}
print("Weights:", {name: str(path) for name, path in weight_paths.items()})
print("SHA-256:", weight_hashes)
encoder = FaceEncoder(WEIGHTS_DIR)
audit, aligned_crops = detect_inventory(audit, encoder)
display(audit[["user_id", "path", "status", "error"]])
audit.to_csv(REPORTS / "image_audit.csv", index=False)
# Inspect alignment and reject images with no face or multiple faces.
aligned_valid = audit[audit.status == "ok"]
fig, axes = plt.subplots(1, len(aligned_valid), figsize=(3 * len(aligned_valid), 3), squeeze=False)
for axis, row in zip(axes.flat, aligned_valid.itertuples()):
    axis.imshow(cv2.cvtColor(aligned_crops[row.source_id], cv2.COLOR_BGR2RGB))
    axis.set_title(row.user_id)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 5. Freeze original-source train / validation / test splits
The seeded split is assigned after auditing but before augmentation or fitting.
Test images and unknown-user images are never used for model selection or
threshold calibration. The manifest makes all assignments inspectable.
With three originals per identity the split is 1/1/1, rather than pretending a
70/15/15 split is meaningful. Add independent captures to grow training data.


In [ ]:
manifest = split_sources(audit, seed=SEED)
manifest.to_csv(PROCESSED / "face_manifest.csv", index=False)
display(pd.crosstab(manifest.user_id, manifest.split))
display(manifest[["user_id", "source_id", "split", "status"]])
print("Underrepresented identities are unknown test people, not enrolled users.")

### 5.1 Save 12 processed images per person from three originals

**Run the next cell to create the actual PNG files on disk.** For each person
with at least three valid independent originals, it takes the first three in
filename order and writes four versions of each: original, +8° rotation,
horizontal flip and grayscale. That is **3 originals + 9 derivatives = 12 files**.

Output: `data/processed/images/<user_id>/` under your configured output root.
Names include the lowercase user ID, source number, original filename stem,
source hash and transform, for example
`maurice__source_01_maurice_neutral__<hash>__grayscale.png`.
The CSV manifest records the original source and its train/validation/test split.
Rerunning with unchanged sources overwrites the same names rather than adding copies.

People with fewer than three original photographs are listed as skipped. With
the current data, Maurice and Larissa each get 12 images; Kudakwashe needs two
more independent photos. Unknown folders are not expanded into enrolled users.

These files are a prepared-image deliverable and visual inspection set.
**Training continues to read the original-source manifest, not this exported
folder.** Derivatives of validation/test originals must not enter training;
they do not count as extra independent evaluation samples.

In [ ]:
from face_auth.preparation import export_processed_images

PROCESSED_IMAGES = PROCESSED / "images"
processed_images, skipped_people = export_processed_images(manifest, PROCESSED_IMAGES)
if len(processed_images):
    counts = processed_images.groupby("user_id").size().rename("saved_images")
    assert counts.eq(12).all(), "Each eligible person must have exactly 12 saved images"
    assert processed_images.groupby("source_id").split.nunique().eq(1).all()
    assert all(Path(path).is_file() for path in processed_images.output_path)
    display(counts.to_frame())
    display(processed_images[["user_id", "augmentation", "split", "output_path"]])
    print(f"Created {len(processed_images)} PNG files in {PROCESSED_IMAGES}")
else:
    print("No eligible people: collect three valid independent originals per person.")
if len(skipped_people):
    print("Skipped people — more original captures may be required:")
    display(skipped_people)
print("Provenance CSV:", PROCESSED / "processed_image_manifest.csv")

# Display the twelve saved files for the first eligible person.
if len(processed_images):
    first_person = processed_images.user_id.iloc[0]
    preview = processed_images[processed_images.user_id == first_person]
    fig, axes = plt.subplots(3, 4, figsize=(13, 10))
    for axis, row in zip(axes.flat, preview.itertuples()):
        with Image.open(row.output_path) as saved_image:
            axis.imshow(saved_image)
        axis.set_title(f"{row.augmentation}\nsource {row.source_id[:8]}")
        axis.axis("off")
    fig.suptitle(f"{first_person}: 3 originals → 12 saved images")
    plt.tight_layout()
    plt.savefig(REPORTS / "processed_image_preview.png", dpi=140, bbox_inches="tight")
    plt.show()


## 6. Extract and save image features
Training gets nine variants (including the original) of each aligned crop.
All descendants retain their original `source_id`. Metadata columns are excluded
from the numeric matrix. Validation and test receive no augmentation.
`image_features.csv` is a sensitive biometric artifact; keep it private.


In [ ]:
features = build_features(manifest, aligned_crops, encoder)
features.to_csv(PROCESSED / "image_features.csv", index=False)
feature_columns = [column for column in features if column.startswith("embedding_")]
assert len(feature_columns) == 128
assert features.groupby("source_id").split.nunique().max() == 1
assert (features.loc[features.split != "train", "augmentation"] == "original").all()
display(features.groupby(["split", "user_id"]).size().rename("feature_rows").to_frame())
print("Saved", len(features), "rows ×", len(feature_columns), "embedding features")

# Only materialize training and validation arrays here. Test remains untouched.
train = features[features.split == "train"]
validation = features[features.split == "validation"]
X_train = train[feature_columns].to_numpy()
y_train = train.user_id.to_numpy()
X_validation = validation[feature_columns].to_numpy()
y_validation = validation.user_id.to_numpy()

## 7. Baseline and classifier training
A dummy prior classifier establishes a baseline. Logistic regression uses
normalized pretrained embeddings rather than learning facial identity from raw
pixels. Compare regularization `C` on validation **only**, selecting macro F1,
then log loss. Augmented images are correlated; their count is not an independent
sample count. Do not interpret training accuracy as generalization evidence.


In [ ]:
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
baseline_prediction = baseline.predict(X_validation)
baseline_metrics = {
    "accuracy": float(accuracy_score(y_validation, baseline_prediction)),
    "macro_f1": float(f1_score(y_validation, baseline_prediction, average="macro", zero_division=0)),
    "log_loss": float(log_loss(y_validation, baseline.predict_proba(X_validation), labels=baseline.classes_)),
}
rows, candidates = [], {}
for C in (0.1, 1.0, 10.0):
    candidate = FaceAuthenticator(C=C, confidence_threshold=CONFIDENCE_THRESHOLD).fit(X_train, y_train)
    candidates[C] = candidate
    rows.append({"C": C, **closed_metrics(candidate, X_validation, y_validation)})
selection = pd.DataFrame(rows).sort_values(["macro_f1", "log_loss", "C"], ascending=[False, True, True])
best_C = float(selection.iloc[0].C)
model = candidates[best_C]
display(pd.DataFrame([baseline_metrics], index=["dummy validation"]))
display(selection)
print("Selected C:", best_C)
display(pd.DataFrame([closed_metrics(model, X_train, y_train),
                      closed_metrics(model, X_validation, y_validation)],
                     index=["training (augmented; optimistic)", "validation (independent originals)"]))

## 8. Validation and authentication threshold
A closed-set classifier always names an enrolled person, including for unknown
faces. Add a cosine gate against the **predicted identity's training centroid**,
in addition to the fixed confidence threshold.

Calibration compares each validation capture to its genuine identity centroid
and other enrolled identities' centroids. Choose a threshold maximizing balanced
genuine acceptance / wrong-identity rejection, using 0.363 (the published SFace
cosine starting point) as a lower bound. Centroids differ from the original
paper's pairwise setup, so this is a documented heuristic, not a security claim.
These wrong-identity comparisons are **not unknown-person false acceptance**.
A separate, larger unknown validation cohort is needed for reliable open-set
calibration. The current unknown cohort remains reserved for final testing.


In [ ]:
calibration = calibrate_similarity(model, X_validation, y_validation)
calibration.to_csv(REPORTS / "validation_threshold_curve.csv", index=False)
fig, axis = plt.subplots()
axis.plot(calibration.threshold, calibration.genuine_acceptance, label="Validation genuine acceptance")
axis.plot(calibration.threshold, calibration.wrong_identity_acceptance, label="Wrong-identity acceptance (proxy)")
axis.axvline(model.similarity_threshold, color="black", linestyle="--", label="Selected cosine gate")
axis.set(xlabel="Cosine threshold", ylabel="Rate", ylim=(-.05, 1.05), title="Validation-only threshold selection")
axis.legend()
plt.tight_layout()
plt.savefig(REPORTS / "threshold_validation.png", dpi=140)
plt.show()
print("Cosine threshold:", model.similarity_threshold, "| confidence threshold:", model.confidence_threshold)
print("Validation authenticated labels:", model.predict(X_validation).tolist())

## 9. Final held-out testing — closed set and unknown rejection
The classifier and thresholds are now frozen; do not tune them after examining
these results. Report accuracy, macro F1 and log loss on original enrolled-person
test images. Log loss uses the underlying normalized classifier probabilities,
not the zeroed authentication rejection scores.

For authentication, report correct identity acceptance, known-person rejection,
wrong identity acceptance, and false acceptance on the entirely unseen people.
The unknown cohort must include real other people; noise/blank images test input
validation rather than unknown-person recognition. With the supplied data, one
unknown photo cannot give a stable false-accept estimate. Missing cohorts produce
`null` metrics, never fabricated zeros.


In [ ]:
test = features[features.split == "test"]
unknown = features[features.split == "unknown_test"]
X_test, y_test = test[feature_columns].to_numpy(), test.user_id.to_numpy()
closed_test = closed_metrics(model, X_test, y_test)
probabilities, _ = model.scores(X_test)
closed_prediction = model.classes_[probabilities.argmax(axis=1)]
auth_prediction = model.predict(X_test)

print("CLOSED-SET TEST (no rejection):")
display(pd.DataFrame([closed_test]))
print(classification_report(y_test, closed_prediction, labels=model.classes_, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, closed_prediction, labels=model.classes_, cmap="Blues")
plt.title("Held-out original images — closed-set identity")
plt.tight_layout()
plt.savefig(REPORTS / "test_confusion_matrix.png", dpi=140)
plt.show()

unknown_predictions = (model.predict(unknown[feature_columns].to_numpy()) if len(unknown)
                       else np.array([], dtype=str))
authentication_test = {
    "known_test_count": int(len(test)), "unknown_test_count": int(len(unknown)),
    "correct_identity_acceptance_rate": float(np.mean(auth_prediction == y_test)),
    "known_rejection_rate": float(np.mean(auth_prediction == "unknown")),
    "known_wrong_identity_acceptance_rate": float(np.mean((auth_prediction != "unknown") & (auth_prediction != y_test))),
    "unknown_false_acceptance_rate": float(np.mean(unknown_predictions != "unknown")) if len(unknown) else None,
}
known_results = test[["user_id", "source_id", "path"]].copy()
known_results["prediction"] = auth_prediction
known_results["cohort"] = "known"
unknown_results = unknown[["user_id", "source_id", "path"]].copy()
unknown_results["prediction"] = unknown_predictions
unknown_results["cohort"] = "unknown"
attempts = pd.concat([known_results, unknown_results], ignore_index=True)
attempts.to_csv(REPORTS / "test_attempts.csv", index=False)
display(attempts[["user_id", "cohort", "prediction"]])
display(pd.DataFrame([authentication_test]))

# A blank image must fail detection, rather than receive an identity prediction.
try:
    encoder.align(np.zeros((256, 256, 3), dtype=np.uint8))
except FaceInputError as error:
    print("Blank-input rejection:", error)
else:
    raise AssertionError("A blank image unexpectedly passed face detection")

## 10. Save artifacts, metadata, metrics and verify reload
Do not refit on validation/test after calibration: the saved classifier is the
same one just evaluated. Metadata on this branch records only the face model and its evaluation.

The `.joblib` file contains the lightweight classifier and centroids. Inference
also needs the `face_auth` package, both ONNX weights and compatible package
versions. Joblib files must only be loaded from trusted sources.


In [ ]:
artifact = MODELS / "face_model.joblib"
joblib.dump(model, artifact)
restored = joblib.load(artifact)
np.testing.assert_allclose(restored.predict_proba(X_test), model.predict_proba(X_test))
assert np.array_equal(restored.predict(X_test), auth_prediction)

# Check that the public face extractor matches stored original-image features.
from feature_extraction import extract_face_features
live_embedding = extract_face_features(test.iloc[0].path, weights_dir=WEIGHTS_DIR)
np.testing.assert_allclose(live_embedding, X_test[:1], atol=1e-6)
print("Reload and shared-extractor parity checks passed.")

metrics = {
    "baseline_validation": baseline_metrics,
    "selected_C": best_C,
    "closed_set_test": closed_test,
    "authentication_test": authentication_test,
    "validation_candidate_results": selection.to_dict(orient="records"),
    "limitations": ["Very small independent sample count", "No liveness or replay protection",
                    "Validation wrong-identity comparisons are not unknown-person FAR",
                    "Single-photo unknown testing is preliminary; collect additional sessions"],
}
(REPORTS / "metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False) + "\n")

metadata_path = MODELS / "model_metadata.json"
metadata = {}
metadata["face"] = {
    "owner": "Maurice", "status": "trained", "threshold": model.confidence_threshold,
    "similarity_threshold": model.similarity_threshold, "selected_C": best_C,
    "classes": model.classes_.tolist(), "feature_columns": feature_columns,
    "pretrained_model": "OpenCV SFace 2021dec + YuNet 2023mar",
    "weights_sha256": weight_hashes,
    "preprocessing": "EXIF-correct RGB; longest side <=1200; BGR; YuNet 0.8; exactly one face; aligned 112x112; L2 SFace 128-D",
    "seed": SEED, "versions": versions, "python": platform.python_version(),
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_counts": {str(k): int(v) for k, v in manifest.split.value_counts().items()},
    "artifact_sha256": sha256(artifact),
    "evaluation": metrics,
}
metadata_path.write_text(json.dumps(metadata, indent=2, allow_nan=False) + "\n")
print("Saved model:", artifact)
print("Saved features:", PROCESSED / "image_features.csv")
print("Saved metadata:", metadata_path)
print("Reports:", REPORTS)

## 11. Interactive face-only authentication demo
Use a new single-face image. An identity label alone is insufficient: both the
classifier confidence and cosine gate must pass. Invalid images are denied.
This does not run or approve the product model: full integration also requires
an enrolled registry mapping and matching voice authentication.


In [ ]:
DEMO_IMAGE = None  # Set to a new authorized or unauthorized image path, then rerun.
if DEMO_IMAGE:
    try:
        embedding = extract_face_features(DEMO_IMAGE, weights_dir=WEIGHTS_DIR)
        label = restored.predict(embedding)[0]
        print("ACCESS DENIED" if label == "unknown" else f"FACE VERIFIED: {label} — voice confirmation still required")
    except FaceInputError as error:
        print("ACCESS DENIED:", error)
else:
    print("Set DEMO_IMAGE to test a new image; held-out test attempts are listed above.")

## 12. Interpretation, next collection steps and references
Before reporting success, inspect `metrics.json`, confusion matrix, per-attempt
results, and source counts. A high score on two known test photos can coexist
with a poor false-accept rate. Repeated runs on the same test set do not add
independent evidence. If you change preprocessing or thresholds after seeing
test results, collect a new test set.

Collect neutral, smiling and surprised images for every member, plus additional
sessions, camera/lighting variations, and separate unknown-person validation and
test cohorts. Review any detector failures and blurry crops. Consider removing
harmful augmentations through validation experiments with more data. Liveness,
spoof/replay defense and privacy controls require separate work.

### Generated deliverables
- `data/processed/images/<user_id>/` — 12 saved PNGs per eligible person
- `data/processed/processed_image_manifest.csv` — saved-image provenance and source splits
- `data/processed/processed_image_skipped.csv` — identities not expanded and reasons
- `data/processed/face_manifest.csv` — audited original-source partitions
- `data/processed/image_features.csv` — provenance plus 128 numeric features
- `models/face_model.joblib` — classifier and rejection gate
- `models/model_metadata.json` — face configuration, provenance and evaluation
- `reports/face/` — audit, validation curve, plots, attempts and JSON metrics

Cloud outputs can be downloaded from the configured writable output root. Keep
biometric features/models private unless all relevant participants consent.

### References
- [OpenCV face detection/recognition tutorial](https://docs.opencv.org/4.11.0/d0/dd4/tutorial_dnn_face.html)
- [OpenCV Zoo YuNet model and license](https://github.com/opencv/opencv_zoo/tree/main/models/face_detection_yunet)
- [OpenCV Zoo SFace model and license](https://github.com/opencv/opencv_zoo/tree/main/models/face_recognition_sface)
- [SFace paper](https://arxiv.org/abs/2205.12010)
